In [4]:
# Assignment #7
# Hands-On with Spark Machine Learning
# You are the owner of an e-commerce web store and you are interested in predicting a customer’s annual e-commerce spend based on historical purchase patterns. In order to develop a linear regression model, execute the following steps:

# Step2. Read in the underlying dataset
# Step3 ~ 9. Clean the “read-in” dataset if needed
# Step11. Split the underlying dataset into “train” and “test” sets
# Step12. Train the ML model on the “train” data set
# Step13. Execute the trained model on the test dataset
# Step14. Compare the output from the ML model to the actual results
# Step15. Examine the efficacy of the ML model using performance metrics covered in the linear regression activity and state the results

##**Step 1**: Install Spark

###**Step 2**: Setting up Spark

Before you can connect to a Spark cluster, Spark needs to be installed. The code below is boilerplate code that can be used to set-up Spark. Please note that this code will be leveraged in all the notebooks since each nodebook is a separate entity.

### **Step 3**. Import the lib




In [5]:
# Colab-friendly setup for PySpark
!apt-get install -y openjdk-11-jdk-headless -qq > /dev/null
!pip install -q "pyspark[connect]==3.5.1" "dataproc-spark-connect==0.8.3"

import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"

from pyspark.sql import SparkSession
spark = SparkSession.builder.master("local[*]").appName("RDDPractice").getOrCreate()
sc = spark.sparkContext

In [6]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


As the owner of an e-commerce web store, you want to predict a customer's annual spending based on their historical purchase behavior. To create a linear regression model for this prediction, follow these steps:

1. **Load the Dataset:** Begin by reading the underlying dataset containing customer purchase data.
2. **Clean the Data:** If necessary, clean the dataset to handle missing values, incorrect entries, or any other inconsistencies.
3. **Split the Data:** Divide the dataset into two subsets: one for training the model ("train" set) and one for testing the model's performance ("test" set).
4. **Train the Model:** Use the "train" dataset to train the machine learning model, allowing it to learn the relationships between the features and the target variable (annual spend).
5. **Evaluate the Model:** Apply the trained model to the "test" dataset to make predictions.
6. **Compare Results:** Compare the model’s predicted values with the actual values in the "test" dataset to assess its accuracy.
7. **Assess Model Performance:** Use the performance metrics from the linear regression analysis (such as R-squared, mean squared error, etc.) to evaluate how well the model performed, and present the results.

By following these steps, you will have developed and evaluated a linear regression model to predict customer spending based on historical data.

# The objective of this exercise is to predict the time on app

##**Step 2**: Read in the data file

In [7]:
# Load in the data
df = spark.read.option("header", "true").csv('/content/drive/MyDrive/Colab Notebooks/UCI/UCI_425.07_Big_Data_Analytics/Assignments/Dataset/Ecomm-Customers-new.csv')

##**Step 3**: Display the contents of the DataFrame

In [8]:
df_new = df['Time on App', 'Time on Website', 'Length of Membership', 'Yearly Amount Spent']

In [9]:
from pyspark.sql.functions import mean

# Calculate the mean value for each column
means = df_new.agg(*(mean(c).alias(c) for c in df_new.columns))

# Replace null values with mean values
data = df_new.na.fill(means.first().asDict())


##**Step 4**: Display the data types

In [10]:
data.dtypes

[('Time on App', 'string'),
 ('Time on Website', 'string'),
 ('Length of Membership', 'string'),
 ('Yearly Amount Spent', 'string')]

In [11]:
# Import all from `sql.types`
from pyspark.sql.types import *


##**Step 5**: Function that converts the data types of the DataFrame columns

In [12]:
# Write a custom function to convert the data type of DataFrame columns
def convertColumn(df, names, newType):
  for name in names:
     df = df.withColumn(name, df[name].cast(newType))
  return df

In [13]:
# Assign all column names to `columns`
columns = ['Time on App','Time on Website','Length of Membership','Yearly Amount Spent']

##**Step 6**: Convert the data types of the above mentioned columns into a float type

In [14]:
from pyspark.sql.types import *
# Conver the `df` columns to `FloatType()`
df = convertColumn(data, columns, FloatType())

##**Step 7**: Confirm that the data type has been converted into float

In [15]:
df.dtypes

[('Time on App', 'float'),
 ('Time on Website', 'float'),
 ('Length of Membership', 'float'),
 ('Yearly Amount Spent', 'float')]

In [16]:
# Print the schema of `df`
df.printSchema()

root
 |-- Time on App: float (nullable = true)
 |-- Time on Website: float (nullable = true)
 |-- Length of Membership: float (nullable = true)
 |-- Yearly Amount Spent: float (nullable = true)



In [17]:
df.describe().show()

+-------+------------------+------------------+--------------------+-------------------+
|summary|       Time on App|   Time on Website|Length of Membership|Yearly Amount Spent|
+-------+------------------+------------------+--------------------+-------------------+
|  count|               500|               500|                 500|                500|
|   mean|12.052487915039062|37.060445365905764|   3.533461554646492|  499.3140381469727|
| stddev|0.9942156264611745|1.0104888427768801|  0.9992775015130736|  79.31478158115246|
|    min|          8.508152|          33.91385|           0.2699011|           256.6706|
|    max|         15.126994|          40.00518|           6.9226894|          765.51843|
+-------+------------------+------------------+--------------------+-------------------+




You should probably standardize your data, as you have seen that the range of minimum and maximum values is quite large.

Your dependent variable is also quite large; you should adjust the values slightly.

##**Step 8**: Processing the data

In [18]:
# Import all from `sql.functions`
from pyspark.sql.functions import *

# Adjust the values of `medianAvatar`
df = df.withColumn("medianTime", col("Time on App")/100000)

# Show the first 2 lines of `df`
df.take(2)

[Row(Time on App=12.655651092529297, Time on Website=39.577667236328125, Length of Membership=4.082620620727539, Yearly Amount Spent=587.9510498046875, medianTime=0.00012655651092529296),
 Row(Time on App=11.109460830688477, Time on Website=37.268959045410156, Length of Membership=2.664034128189087, Yearly Amount Spent=392.2049255371094, medianTime=0.00011109460830688477)]

In [19]:
# Re-order and select columns
df = df.select("Time on App",
               "Time on Website",
               "Length of Membership",
               "Yearly Amount Spent")

In [20]:
df.show(10)

+-----------+---------------+--------------------+-------------------+
|Time on App|Time on Website|Length of Membership|Yearly Amount Spent|
+-----------+---------------+--------------------+-------------------+
|  12.655651|      39.577667|           4.0826206|          587.95105|
|  11.109461|       37.26896|           2.6640341|          392.20493|
|  11.330278|      37.110596|            4.104543|          487.54752|
|  13.717514|      36.721283|           3.1201787|          581.85236|
|  12.795189|       37.53665|            4.446308|          599.40607|
|  12.026925|       34.47688|           5.4935074|           637.1025|
|  11.366348|      36.683777|            4.685017|           521.5722|
|  12.351959|       37.37336|           4.4342732|           549.9042|
|  13.386235|      37.534496|           3.2734337|          570.20044|
|  11.814128|       37.14517|            3.202806|          427.19937|
+-----------+---------------+--------------------+-------------------+
only s

##**Step 9**: Specifying the label and the features

In [21]:
# Import `DenseVector`
# A Dense Vector is used to store arrays of values for use in PySpark.
from pyspark.ml.linalg import DenseVector

# # Define the `input_data`
# input_data = df.rdd.map(lambda x: (x[0], DenseVector(x[1:])))


input_data = df.rdd.map(lambda x: (x[0], DenseVector(x[1:])))

# Replace `df` with the new DataFrame
df = spark.createDataFrame(input_data, ["label", "features"])

label = df.rdd.map(lambda x: x.label)
features = df.rdd.map(lambda x: x.features)

##**Step 10**: Scaling the features
using 'StandardScaler' - standardizes a feature of the model by subtracting the mean and then scaling to unit variance. Unit variance means dividing all the values by the standard deviation.**


In [22]:
# Import `StandardScaler`
from pyspark.ml.feature import StandardScaler

# Initialize the `standardScaler`
standardScaler = StandardScaler(inputCol="features", outputCol="features_scaled")

# Fit the DataFrame to the scaler
scaler = standardScaler.fit(df.select('features'))

# Transform the data in `df` with the scaler
scaled_df = scaler.transform(df)

# Inspect the result
scaled_df.take(2)

[Row(label=12.655651092529297, features=DenseVector([39.5777, 4.0826, 587.951]), features_scaled=DenseVector([39.1669, 4.0856, 7.4129])),
 Row(label=11.109460830688477, features=DenseVector([37.269, 2.664, 392.2049]), features_scaled=DenseVector([36.8821, 2.666, 4.9449]))]

##**Step 11**: Create the "Train/Test" split

In [23]:
# Split the data into train and test sets
train_data, test_data = scaled_df.randomSplit([.7,.3],seed=1234)

##**Step 12**: Train the ML model on the “train” data set

In [24]:
# Import `LinearRegression`
from pyspark.ml.regression import LinearRegression

# Initialize `lr`
lr = LinearRegression(labelCol="label", maxIter=100, regParam=0.3, elasticNetParam=0.8)

In [25]:
linearModel_cus = lr.fit(train_data)

##**Step 13**: Make the predictions

In [26]:
# Make predictions on test data
predicted = linearModel_cus.transform(test_data)

In [27]:
# Retrieve the predictions and the "known" labels
predictions = predicted.select("prediction").rdd.map(lambda x: x[0])
labels = predicted.select("label").rdd.map(lambda x: x[0])


In [28]:
# Combine the predictions and the label
predictionAndLabel = predictions.zip(labels).collect()

##**Step 14**: Output the predictions and the associated labels

In [29]:
# Print out first 5 instances of `predictionAndLabel` - Please note that the medianHouseValue was divided by 100000 in Step 17
predictionAndLabel[:15]

[(11.556771930943693, 8.50815200805664),
 (11.573011439284393, 8.668349266052246),
 (12.082074870408434, 9.607315063476562),
 (11.810025613072094, 10.34787654876709),
 (12.011664963007144, 10.480506896972656),
 (11.7522242541732, 10.534553527832031),
 (11.971161194334769, 10.537307739257812),
 (11.838157674066203, 10.542645454406738),
 (12.160741600968091, 10.572466850280762),
 (11.965697200676683, 10.610536575317383),
 (11.864690961758555, 10.627948760986328),
 (11.853882059478746, 10.674653053283691),
 (11.629550012013022, 10.75713062286377),
 (11.8706007228443, 10.771074295043945),
 (11.929562828535612, 10.869163513183594)]

##**Step 15**: Evaluating the model

**RMSE**: RMSE measures the differences between predicted values by the model and the actual values.The smaller the RMSE value is, the closer predicted and actual values are.

In [30]:
linearModel_cus.summary.rootMeanSquaredError

0.8985157150475345

The model performance is low as 89% root mean square

In [31]:
linearModel_cus.summary.r2

0.1517756876513976

The R2 linear coefficient is small of 16.8%. The model of feature almost have no correlation coefficient with the dv (amount spent on e-commercial yearly)